In [20]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [21]:
words = open('names.txt', 'r').read().splitlines()
print(words[:8], f'... len: {len(words)}')

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'] ... len: 32033


In [22]:
chrs = sorted(list(set(''.join(words))))
stoi = {s : i+1 for i, s in enumerate(chrs)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}

In [23]:
# dataset
block_size = 3
X, Y = [], []
for w in words[:5]:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [24]:
X.shape

torch.Size([32, 3])

In [25]:
X[:5] # [0, 5, 13] is gonna be one of our input, so all of them will have their own coordinates(this is called embadded)

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])

In [26]:
C = torch.randn((27, 2)) # this contains all coordinates (vectors) of our embadded characters (from '.' - 0 and 'a' - 1 to 'z' - 26)

In [27]:
""" 
C[X[:5]] replaces every index in X[:5] with its row from C (its embedding vector)
  shape: (5, 3, 2)
    5 - examples (rows of X[:5])
    3 - characters in each example (context length)
    2 - size of each embedding vector 
"""
e = C[X[:5]] # we get the same result, but now every number in a row has its own embedded value
# so to store this 2d arrays we need one more demension: 5 - rows from the original array | 3 - lenghts of the rows in the original array | 2 - their embedded values
# thus every number gets its embaddened magnitude 
e

tensor([[[ 1.0504, -0.2277],
         [ 1.0504, -0.2277],
         [ 1.0504, -0.2277]],

        [[ 1.0504, -0.2277],
         [ 1.0504, -0.2277],
         [ 0.4744, -0.0596]],

        [[ 1.0504, -0.2277],
         [ 0.4744, -0.0596],
         [ 1.4173, -1.0595]],

        [[ 0.4744, -0.0596],
         [ 1.4173, -1.0595],
         [ 1.4173, -1.0595]],

        [[ 1.4173, -1.0595],
         [ 1.4173, -1.0595],
         [-0.2040, -0.2976]]])

In [28]:
e[:, 0, :] # C[X[:5][:, 0, :]

tensor([[ 1.0504, -0.2277],
        [ 1.0504, -0.2277],
        [ 1.0504, -0.2277],
        [ 0.4744, -0.0596],
        [ 1.4173, -1.0595]])

In [29]:
torch.unbind(C[X[:5]], 1) # [:, 0, :] & [:, 1, :] & [:, 2, :]

(tensor([[ 1.0504, -0.2277],
         [ 1.0504, -0.2277],
         [ 1.0504, -0.2277],
         [ 0.4744, -0.0596],
         [ 1.4173, -1.0595]]),
 tensor([[ 1.0504, -0.2277],
         [ 1.0504, -0.2277],
         [ 0.4744, -0.0596],
         [ 1.4173, -1.0595],
         [ 1.4173, -1.0595]]),
 tensor([[ 1.0504, -0.2277],
         [ 0.4744, -0.0596],
         [ 1.4173, -1.0595],
         [ 1.4173, -1.0595],
         [-0.2040, -0.2976]]))

`W1 = torch.randn((6, 100))` -- this is gonna be our first hidden layer; \
therefore the problem is that C[X[:5]].shape = (5, 3, 2), so we need to somehow concatenate 3 and 2 together to make .shape equal to (5, 6)

In [30]:
torch.cat([C[X[:5]][:, 0, :], C[X[:5]][:, 1, :], C[X[:5]][:, 2, :]], 1) # always needs to be changed if we change the block_size variable
# or
torch.cat(torch.unbind(e, 1), 1) # universal approach

tensor([[ 1.0504, -0.2277,  1.0504, -0.2277,  1.0504, -0.2277],
        [ 1.0504, -0.2277,  1.0504, -0.2277,  0.4744, -0.0596],
        [ 1.0504, -0.2277,  0.4744, -0.0596,  1.4173, -1.0595],
        [ 0.4744, -0.0596,  1.4173, -1.0595,  1.4173, -1.0595],
        [ 1.4173, -1.0595,  1.4173, -1.0595, -0.2040, -0.2976]])

But the concatenate operation is very inefficient. \
The solution is to use PyTorch's .view() function.

In [31]:
C[X[:5]].view(5, 6)

tensor([[ 1.0504, -0.2277,  1.0504, -0.2277,  1.0504, -0.2277],
        [ 1.0504, -0.2277,  1.0504, -0.2277,  0.4744, -0.0596],
        [ 1.0504, -0.2277,  0.4744, -0.0596,  1.4173, -1.0595],
        [ 0.4744, -0.0596,  1.4173, -1.0595,  1.4173, -1.0595],
        [ 1.4173, -1.0595,  1.4173, -1.0595, -0.2040, -0.2976]])

___

### model

#### preparations

In [32]:
import torch
import torch.nn.functional as F
import random

In [33]:
words = open('names.txt', 'r').read().splitlines()

chrs = sorted(list(set(''.join(words))))
stoi = {s : i+1 for i, s in enumerate(chrs)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}
print(stoi)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [34]:
# dataset
X, y = [], []
block_size = 3

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
y = torch.tensor(y)

X.shape, y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [35]:
# train / valid / test
block_size = 4
embedded_dim = 5
def build_dataset(words):
    X, y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    y = torch.tensor(y)
    print(X.shape, y.shape)
    return X, y

random.seed(52)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

X_train, y_train = build_dataset(words[:n1])
X_val, y_val = build_dataset(words[n1:n2])
X_test, y_test = build_dataset(words[n2:])

torch.Size([182498, 4]) torch.Size([182498])
torch.Size([22641, 4]) torch.Size([22641])
torch.Size([23007, 4]) torch.Size([23007])


In [36]:
# parameters 
gen = torch.Generator().manual_seed(52)
dim_mul = embedded_dim * block_size
C = torch.rand((27, embedded_dim), generator=gen)
# first hidden layer with tanh()
W1 = torch.randn((dim_mul, 200), generator=gen)
b1 = torch.randn(200, generator=gen)
# output_layer
W2 = torch.randn((200, 27), generator=gen)
b2 = torch.rand(27, generator=gen)

parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

print(f'num of parameters: {sum(p.nelement() for p in parameters)}')

num of parameters: 9762


In [ ]:
# emb = C[X] 
# # torch.cat(torch.unbind(emb, 1), 1) 
# hidden_layer = torch.tanh(emb.view(-1, 6) @ W1 + b1) # it is good to check broadcasting here
# # (emb.view(-1, 6) @ W1).shape
# # b1.shape
# logits = hidden_layer @ W2 + b2
# # classification
# # counts = logits.exp()
# # probs = counts / counts.sum(1, keepdim=True)
# # loss = -probs[torch.arange(32), Y].log().mean()
# loss = F.cross_entropy(logits, y)
# loss

#### learning

In [45]:
for _ in range(10):

    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (32,))

    # forward pass
    emb = C[X_train[ix]]
    hidden_layer = torch.tanh(emb.view(-1, dim_mul) @ W1 + b1)
    logits = hidden_layer @ W2 + b2
    loss = F.cross_entropy(logits, y_train[ix])
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    lr = 0.01
    for p in parameters:
        p.data -= lr * p.grad

print(loss.item(), logits.shape, hidden_layer.shape)

13.101205825805664 torch.Size([32, 27]) torch.Size([32, 200])


In [39]:
emb = C[X_train]
h = torch.tanh(emb.view(-1, dim_mul) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, y_train)
loss

tensor(16.7998, grad_fn=<NllLossBackward0>)

In [40]:
with torch.no_grad():
    emb = C[X_val]
    h = torch.tanh(emb.view(-1, dim_mul) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y_val)
loss

tensor(16.7525)

In [41]:
with torch.no_grad():
    emb = C[X_test]
    h = torch.tanh(emb.view(-1, dim_mul) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y_test)
loss

tensor(16.7656)

In [42]:
# gen = torch.Generator().manual_seed(52)

for _ in range(10):

    out = []
    context = [0] * block_size

    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        out.append(ix)
        context = context[1:] + [ix]
        if ix == 0:
            break
    print(''.join(itos[i] for i in out))

iaaqxalqizipyihyqqnpzcvqpzzqfndzhiizzxqpplzqfzlqopazxvfpztxppzxxvfpzpxyqpalqxqlzlqzqlqpplqxpldhilzidqpazqpzlqopazxvfpztdppvfxyjlqxqcplzqfzlqopazxqfplzqfzlqopazxvfpzpzvfnpzixppcpxcqxplzqfzl.
zilqpalqxqlzlqfclzidqpalqxclpopawhzxqpplzqfzlqopapxpxwqpalqxqlzlqzqlqopazxpzddqpalqoiazxvfpzpzyqeqinlhiizzxqpplzhizizvfqcpppzwdpphfxwpzlqopapxyqxalpopawxpxcqpplzqfzlqopazxvfpzpxvfnpzwzqpalqxclpopapxvfndphvxpzzqfndzhiipzvxqizlqxqlzlqzqlqpplzqpzlqopazxpzcqfplzhizwpqpxlqxqlzlqzqlqh.
zvfizzxqpplzqfzlqopazxpzqftlhopazxqfplzqfzlqopazvfzzqfndzhiipzvqpzlqopapxyqhilzjqfplzhdzizvfxpqpxlqxplzqfzlqopapxvfncpicxxxilaaqzalqpclqxqlzlqzqlqpplzqpzlqopapxpxcqpclzldaqizwpqpxlqopazxpadpzilvzxlqpnlhirazxvfpzzqfndzhiizcxvpncppcpplqxplzqpzlqopazxpzdqpclqxqlzlqzqlqhplzidqpalzyqfplzqfzlqopapxyqxplzqfzlqopazxpzcqfplqxnlzlvfcztlqhtlqizlqxqlzlqzqlqppcphwqqpcpzwqpplzqfzlqopazqpzlqopazxpzcqfplzqfzlqopazxpzilajqzxlqpnchilziqfcltopazxvfpzzqfndzhizizvfzzqeqpnlhizcpadpzilajqfxlzidqpalhiizzxqqplzqfzlqopazxdqpalqxclzidppazhdqpalqoilzx

In [43]:
logits

tensor([[ 12.6273,   7.5112, -12.1903,  -3.6835,   9.6176,  12.6371,  -4.9661,
           1.4450,   7.6590,   7.5532,   0.7193,  -0.9364,   4.9814,   7.2789,
          11.4455,   4.0336,  14.4669,  16.1322,   2.9704, -13.5469,   2.1323,
          -6.0354,   3.0369, -16.1437,   9.0985,   9.3060,  12.3303]],
       grad_fn=<AddBackward0>)

In [44]:
probs

tensor([[2.3380e-02, 1.4026e-04, 3.8967e-13, 1.9282e-09, 1.1529e-03, 2.3610e-02,
         5.3474e-10, 3.2541e-07, 1.6261e-04, 1.4629e-04, 1.5751e-07, 3.0075e-08,
         1.1176e-05, 1.1120e-04, 7.1719e-03, 4.3320e-06, 1.4716e-01, 7.7804e-01,
         1.4960e-06, 1.0036e-13, 6.4705e-07, 1.8356e-10, 1.5989e-06, 7.4781e-15,
         6.8602e-04, 8.4415e-04, 1.7372e-02]], grad_fn=<SoftmaxBackward0>)